# Mini-projet : Construire un chatbot IA local avec mémoire


👩‍🏫 👩🏿‍🏫 Ce que vous apprendrez
Comment exécuter un modèle de langage volumineux (LLM) quantifié localement à l'aide de llama.cpp.
Comment créer une interface de chatbot interactive en utilisant Streamlit.
Comment utiliser LlamaCpple wrapper de LangChain pour intégrer des modèles dans une application.
Comment implémenter la mémoire à court terme à l'aide de LangChain ConversationBufferMemory.
Comment concevoir la personnalité de votre chatbot à l'aide d'une invite système.


🛠️ Ce que vous allez créer
Un chatbot local entièrement fonctionnel qui :

Utilise le modèle quantifié Mistral 7B de Hugging Face,
Comprend le contexte des messages précédents et répond en conséquence,
Offre une interface utilisateur avec un flux conversationnel clair,
Vous permet de définir la personnalité de votre propre assistant (par exemple, « un pirate sarcastique », « un tuteur concis »).


Qu'allez-vous utiliser
Concepts :

Inférence LLM quantifiée avecllama.cpp
Mémoire conversationnelle via LangChainConversationBufferMemory
Ingénierie rapide avec messages système
Streaming de jetons en temps réel pour l'UX
Bibliothèques et outils :

llama-cpp-python(pour exécuter le modèle GGUF quantifié localement)
langchain(pour les modèles d'invite, la mémoire et les chaînes)
streamlit(pour l'interface utilisateur Web et les composants de chat)
huggingface_hub(pour récupérer les artefacts du modèle)
Modèle:

Mistral-7B-Instruct-v0.1.Q4_0.gguf(quantifié, à partir du dépôt HF de TheBloke)

# Analyse approfondie de l’énoncé et **approche de résolution** — Mini-projet : Chatbot IA local avec mémoire

## 1. **Analyse détaillée de l’énoncé**

L’objectif du projet est de **développer un chatbot IA local** capable de retenir le contexte d’une conversation (mémoire à court terme), personnalisable par l’utilisateur, et s’appuyant sur un modèle LLM quantifié Mistral 7B (format GGUF) fonctionnant en local (llama.cpp).
Le tout doit être exposé via une interface Web interactive (Streamlit) et intégrer la gestion de la mémoire conversationnelle grâce à LangChain.

Les **points durs** du projet :

* Exécution locale du modèle quantifié (optimisation de la mémoire/CPU, gestion des dépendances natives).
* Intégration de la mémoire de conversation pour la contextualisation des échanges.
* Dynamisme et persistance de l’interface (historique, gestion d’état utilisateur).
* Personnalisation du comportement de l’IA (système prompt dynamique).
* Streaming de la génération pour une UX fluide.
* Modularité du pipeline (prompt > mémoire > modèle > restitution).

---

## 2. **Approche de résolution simple, robuste et fonctionnelle**

### a) **Structuration du projet et environnement**

* **Création d’un dossier de projet isolé**.
* **Environnement virtuel** pour éviter tout conflit de dépendances.
* Installation stricte des packages nécessaires :

  * `llama-cpp-python` (backend modèle quantifié)
  * `streamlit` (frontend web)
  * `langchain` (gestion de chaînes, prompts, mémoire)
  * `huggingface_hub` (téléchargement programmatique du modèle)
  * (Optionnel) `tqdm`, `psutil` pour des usages avancés (pas requis au début)

### b) **Interface utilisateur (Streamlit)**

* **Page principale** : entête, explications courtes.
* **Barre latérale** : zone de saisie de l’invite système (permet à l’utilisateur de définir la personnalité du bot).
* **Zone de chat** :

  * Utilisation de `st.chat_message` pour afficher l’historique des échanges.
  * Utilisation de `st.chat_input` pour l’entrée utilisateur.
* **Session State** : stockage de l’historique (messages utilisateur/assistant) pour gestion du contexte.

### c) **Chargement du modèle LLM**

* **Téléchargement automatisé** via `huggingface_hub` si le fichier modèle n’est pas déjà présent.
* **Chargement avec llama.cpp** :

  * Attention à la configuration des paramètres (chemin modèle, température, nombre max de tokens, streaming activé/désactivé).
  * Utilisation du wrapper LangChain pour simplifier l’appel.

### d) **Pipeline de prompt et gestion du contexte**

* **Template de prompt** :

  * Inclure :

    * Message système (défini par l’utilisateur)
    * Historique des échanges (via mémoire LangChain)
    * Dernière entrée utilisateur
    * Espace pour la réponse de l’assistant
* **Gestion de la mémoire** :

  * Usage de `ConversationBufferMemory` (mémoire tampon à court terme).
  * Clé mémoire dédiée (ex : `"chat_history"`).
  * Mise à jour manuelle de la mémoire après chaque interaction.

### e) **Chaînage LangChain**

* Création d’une **pipeline** qui :

  * Récupère l’historique et la nouvelle entrée utilisateur.
  * Construit l’invite finale.
  * Passe au modèle pour génération.
  * Met à jour la mémoire.
  * Retourne la réponse à afficher.

### f) **Streaming (facultatif mais recommandé)**

* Utilisation du mode `streaming=True` de llama.cpp pour afficher la réponse en temps réel (meilleure UX).
* Boucle de collecte de jetons et affichage progressif.

### g) **Boucle de dialogue**

* À chaque message utilisateur :

  * Ajout du message à l’historique.
  * Passage dans le pipeline LLM.
  * Ajout de la réponse IA à l’historique.
  * Affichage complet de l’historique (avec distinction rôles).
* **Reset session** possible pour recommencer une conversation propre.

---

## 3. **Points de robustesse et de simplicité**

* **Robustesse** :

  * Toutes les dépendances isolées dans l’environnement virtuel.
  * Téléchargement automatique du modèle.
  * Validation de la présence du modèle avant initialisation.
  * Gestion d’exceptions lors des appels modèle et affichages d’erreurs clairs.
  * Historique dans `st.session_state` = robustesse UX.
* **Simplicité** :

  * Code découpé en fonctions claires (chargement modèle, création prompt, appel LLM, gestion mémoire, rendu Streamlit).
  * Pas de dépendances inutiles ou d’options complexes en première passe.
  * Personnalisation simple du prompt système.
* **Extensibilité** :

  * Possibilité d’ajouter d’autres types de mémoire.
  * Ajout d’une sauvegarde/chargement d’historique.

---

## 4. **Synthèse des étapes (pipeline de résolution)**

1. **Initialiser l’environnement Python et installer les dépendances.**
2. **Créer l’interface utilisateur Streamlit avec gestion d’historique et d’invite système.**
3. **Télécharger et charger le modèle Mistral 7B quantifié via llama.cpp.**
4. **Définir le template de prompt, gérer l’historique avec ConversationBufferMemory.**
5. **Construire le pipeline LangChain (input utilisateur + historique -> prompt -> LLM -> mémoire -> output).**
6. **Gérer le flux de chat avec affichage dynamique et streaming des réponses.**
7. **Tester différentes personnalités et scénarios, valider la persistance du contexte.**

---

## 5. **Conseils**

* **Travailler d’abord sur la version sans streaming pour valider le pipeline, puis ajouter le streaming.**
* **Décomposer chaque composant pour faciliter les tests et le debug.**
* **Documenter chaque fonction et étape clé pour pouvoir étendre facilement le chatbot (mémoires avancées, sauvegarde, API, etc).**

---


# Cellule 1 : Installation et import des librairies

In [ ]:
# Installer les dépendances nécessaires (à exécuter UNE FOIS dans le terminal ou en cellule Jupyter magique)
# !pip install streamlit langchain llama-cpp-python huggingface_hub

# Imports principaux
import os
import streamlit as st
from huggingface_hub import hf_hub_download
from langchain.llms import LlamaCpp
from langchain.memory import ConversationBufferMemory
from langchain.prompts import ChatPromptTemplate
from langchain.chains import ConversationChain


# Cellule 2 : Téléchargement automatique du modèle GGUF

In [ ]:
# Paramètres du modèle : repo HF et nom du fichier (modèle Mistral 7B quantifié)
MODEL_REPO = "TheBloke/Mistral-7B-Instruct-v0.1-GGUF"
MODEL_FILE = "mistral-7b-instruct-v0.1.Q4_0.gguf"

# Dossier local pour stocker le modèle
MODEL_DIR = "./models"

# Crée le dossier si nécessaire
os.makedirs(MODEL_DIR, exist_ok=True)

# Chemin absolu du modèle
model_path = os.path.join(MODEL_DIR, MODEL_FILE)

# Téléchargement (fait une seule fois)
if not os.path.exists(model_path):
    print("Téléchargement du modèle, veuillez patienter (environ 4 Go)...")
    hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir=MODEL_DIR)
    print("Modèle téléchargé.")
else:
    print("Modèle déjà téléchargé.")



# Cellule 3 : Initialisation du modèle LLM avec llama-cpp

In [ ]:
# Chargement du modèle avec les bons paramètres (quantification Q4_0, streaming activé)
llm = LlamaCpp(
    model_path=model_path,
    temperature=0.7,              # Contrôle la diversité des réponses
    max_tokens=512,               # Longueur max d'une réponse
    streaming=True,               # Active le streaming pour une meilleure UX
    n_ctx=2048                    # Taille du contexte (doit correspondre à la RAM dispo)
)


# Cellule 4 : Mise en place de la mémoire conversationnelle

In [ ]:
# Mémoire conversationnelle : conserve l'historique complet du dialogue
memory = ConversationBufferMemory(
    memory_key="chat_history",        # clé d'accès à la mémoire
    return_messages=True
)


# Cellule 5 : Template du prompt et chaîne LangChain

In [ ]:
# Prompt dynamique avec place pour :
# - le message système (personnalité, rôle du bot)
# - l'historique des échanges
# - la nouvelle entrée utilisateur

# Utilisation du ChatPromptTemplate pour moduler l'attitude du chatbot
system_template = (
    "{system_prompt}\n"
    "Contexte de la conversation :\n"
    "{chat_history}\n"
    "Utilisateur : {input}\n"
    "Assistant :"
)

prompt = ChatPromptTemplate.from_template(system_template)

# Création de la chaîne de dialogue LangChain (pipeline)
chat_chain = ConversationChain(
    llm=llm,
    prompt=prompt,
    memory=memory,
    input_key="input"
)


# Cellule 6 : Interface utilisateur avec Streamlit

In [ ]:
# Configuration de la page Streamlit
st.set_page_config(page_title="Chatbot IA local avec mémoire", page_icon=None)
st.title("🤖 Chatbot IA Local – Mistral 7B + Mémoire")
st.write("Chattez avec un LLM local doté de mémoire conversationnelle.")

# Barre latérale pour personnaliser la personnalité du bot
default_system_prompt = "Vous êtes un assistant utile, concis et poli."
system_prompt = st.sidebar.text_area(
    "Personnalité du chatbot (prompt système)",
    value=default_system_prompt,
    height=120
)

# Initialisation ou récupération de l'historique de chat
if "messages" not in st.session_state:
    st.session_state.messages = []

# Affichage de l'historique dans l'ordre
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Saisie utilisateur
user_input = st.chat_input("Tapez votre message ici...")

if user_input:
    # Ajout du message utilisateur à l'historique
    st.session_state.messages.append({"role": "user", "content": user_input})

    # Génération de la réponse du bot
    with st.chat_message("assistant"):
        # Construction du prompt complet
        inputs = {
            "input": user_input,
            "system_prompt": system_prompt
        }
        # Appel de la chaîne (streaming activé)
        response = chat_chain.run(inputs)
        st.markdown(response)

    # Ajout de la réponse à l'historique
    st.session_state.messages.append({"role": "assistant", "content": response})


# Cellule 7 : Lancement de l’application

In [ ]:
# (À exécuter dans un terminal pour lancer Streamlit, PAS dans une cellule Python classique)
# !streamlit run <nom_du_script>.py

# Exemple :
# !streamlit run chatbot_local.py
